# NLP â€” rule-based parser and structured inference

This notebook inspects the implemented parser and baseline inference schema on existing master records. It displays evidence, uncertainty, and structured fields inline without running the writer-oriented CLI.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display

current = Path.cwd().resolve()
ROOT = next(
    (p for p in (current, *current.parents) if (p / "src" / "liversira_ai").is_dir()),
    None
)

SRC_DIR = ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print({"project_root": str(ROOT), "src_dir": str(SRC_DIR)})

from liversira_ai.nlp.parser import parse_liver_report, split_sentences
from liversira_ai.nlp.baseline import extract_liver_lesion_features

MASTER_PATH = ROOT / 'data/processed/nlp/master_records.jsonl'
records = [json.loads(line) for line in MASTER_PATH.open(encoding='utf-8') if line.strip()]
print({'master_records': len(records), 'first_id': records[0]['id']})

{'project_root': 'D:\\University\\Graduation Project\\LiverSira AI', 'src_dir': 'D:\\University\\Graduation Project\\LiverSira AI\\src'}


{'master_records': 9262, 'first_id': 'BDMAP_00000001'}


## Parser examples and evidence

In [2]:
examples = []
for record in records:
    text = record.get('reports', {}).get('narrative_clean')
    if text:
        parsed = parse_liver_report(text)
        if parsed.presence is not None:
            examples.append((record['id'], text, parsed))
    if len(examples) == 3:
        break
for record_id, text, parsed in examples:
    print(record_id)
    display(pd.DataFrame([{
        'sentences': split_sentences(text)[:3], 'presence': parsed.presence,
        'presence_reason': parsed.presence_reason, 'lesion_count': parsed.lesion_count,
        'location_segments': parsed.location_segments, 'dimensions_cm': parsed.reported_dimensions_cm,
        'max_diameter_cm': parsed.derived_max_diameter_cm, 'attenuation': parsed.attenuation,
    }]))
    display(pd.DataFrame([item.__dict__ for item in parsed.evidence]))

BDMAP_00000001


,sentences,presence,presence_reason,lesion_count,location_segments,dimensions_cm,max_diameter_cm,attenuation
0,"[The patient has a normal-sized spleen, measur...",True,None,25,[2],"[3.1, 1.6]",3.1,hypoattenuating


,target_field,text,start,end
0,liver_lesions.has_lesion,"However, there are multiple hypoattenuating li...",209,314
1,liver_lesions.has_lesion,The largest lesion is located in hepatic segme...,315,392
2,liver_lesions.has_lesion,The total volume of all liver masses is 19.4 cc.,393,441
3,liver_lesions.has_lesion,IMPRESSION: Multiple hypoattenuating liver mas...,741,872
4,liver_lesions.has_lesion,The total volume of all liver masses is 19.4 cc.,873,921
5,liver_lesions.lesion_count,total of 25 lesions,286,305
6,liver_lesions.largest_lesion.location_segments,hepatic segment 2 and,348,369
7,liver_lesions.largest_lesion.location_segments,hepatic segment 2,854,871
8,liver_lesions.largest_lesion.reported_dimensio...,largest lesion measuring 3.1 x 1.6 cm,813,850
9,liver_lesions.largest_lesion.derived_max_diame...,largest lesion measuring 3.1 x 1.6 cm,813,850


BDMAP_00000004


,sentences,presence,presence_reason,lesion_count,location_segments,dimensions_cm,max_diameter_cm,attenuation
0,"[The spleen, liver, pancreas, and kidneys are ...",True,None,None,None,None,None,None


,target_field,text,start,end
0,liver_lesions.has_lesion,RESULTS: - Spleen size and enhancement: normal...,793,1015
1,liver_lesions.largest_lesion.reported_dimensio...,measuring 3.4 x 2.9 cm,619,641
2,liver_lesions.largest_lesion.derived_max_diame...,measuring 3.4 x 2.9 cm,619,641
3,liver_lesions.largest_lesion.reported_dimensio...,measuring 3.4 x 2.9 cm,978,1000
4,liver_lesions.largest_lesion.derived_max_diame...,measuring 3.4 x 2.9 cm,978,1000
5,liver_lesions.largest_lesion.reported_dimensio...,measuring 3.4 x 2.9 cm,1046,1068
6,liver_lesions.largest_lesion.derived_max_diame...,measuring 3.4 x 2.9 cm,1046,1068
7,liver_lesions.largest_lesion.attenuation,hypoattenuating,707,722
8,liver_lesions.largest_lesion.attenuation,hypoattenuating,1018,1033


BDMAP_00000005


,sentences,presence,presence_reason,lesion_count,location_segments,dimensions_cm,max_diameter_cm,attenuation
0,"[The spleen is normal in size, measuring 73.8 ...",False,None,None,None,None,None,None


,target_field,text,start,end
0,liver_lesions.has_lesion,IMPRESSION: No evidence of tumor is observed i...,485,565


## Structured baseline output

In [3]:
rows = []
for record in records[:200]:
    text = record.get('reports', {}).get('narrative_clean')
    features = extract_liver_lesion_features(text)
    rows.append({'id': record['id'], **{key: features[key] for key in ['has_lesion', 'lesion_count', 'location_segments', 'derived_max_diameter_cm', 'attenuation']}, 'unclear_fields': len(features['missing_or_unclear']), 'evidence_spans': len(features['evidence'])})
inference_table = pd.DataFrame(rows)
display(inference_table.head(10))
display(inference_table[['has_lesion', 'unclear_fields', 'evidence_spans']].value_counts().rename('records').reset_index())

,id,has_lesion,lesion_count,location_segments,derived_max_diameter_cm,attenuation,unclear_fields,evidence_spans
0,BDMAP_00000001,True,25.0,[2],3.1,hypoattenuating,0,16
1,BDMAP_00000002,None,NaN,None,NaN,NaN,1,0
2,BDMAP_00000003,None,NaN,None,NaN,NaN,1,6
3,BDMAP_00000004,True,1.0,None,NaN,NaN,0,9
4,BDMAP_00000005,False,0.0,None,NaN,NaN,0,1
5,BDMAP_00000006,True,2.0,[8],0.6,hypoattenuating,0,21
6,BDMAP_00000007,False,0.0,None,NaN,NaN,0,1
7,BDMAP_00000008,None,NaN,None,NaN,NaN,1,7
8,BDMAP_00000009,True,31.0,[5],2.9,hypoattenuating,0,27
9,BDMAP_00000010,None,NaN,None,NaN,NaN,1,7


,has_lesion,unclear_fields,evidence_spans,records
0,False,0,1,35
1,True,0,16,7
2,True,0,27,4
3,True,0,11,4
4,True,0,15,4
5,True,0,22,3
6,True,0,14,3
7,True,0,13,2
8,True,0,7,2
9,True,0,20,2


## Existing JSONL schema and provenance inspection

In [4]:
ground_truth_path = ROOT / 'data/processed/nlp/ground_truth.jsonl'
inference_path = ROOT / 'results/nlp/predictions/baseline_inferences_updated.jsonl'
gt_first = json.loads(next(ground_truth_path.open(encoding='utf-8')))
inf_first = json.loads(next(inference_path.open(encoding='utf-8')))
display(pd.DataFrame([{'ground_truth_keys': list(gt_first), 'inference_keys': list(inf_first), 'inference_input_field': inf_first['input_reference']['field'], 'inference_confidence': inf_first['confidence']}]))
print({'ground_truth_status': gt_first['validation']['status'], 'evidence_span_count': len(inf_first.get('evidence_spans', [])), 'missing_or_unclear_count': len(inf_first.get('missing_or_unclear', []))})

,ground_truth_keys,inference_keys,inference_input_field,inference_confidence
0,"[id, ground_truth, validation]","[id, input_reference, study, liver_lesions, ev...",narrative_clean,None


{'ground_truth_status': 'pending', 'evidence_span_count': 16, 'missing_or_unclear_count': 0}


## Summary

The parser preserves uncertainty instead of treating absent evidence as a negative. Baseline records carry structured fields, evidence spans, and explicit missing/unclear fields. This notebook inspects those outputs and does not alter inference behavior or regenerate JSONL files.

In [5]:
assert len(records) == 9262
assert inf_first['input_reference']['field'] == 'narrative_clean'
print('Parser and structured inference inspection completed without writing files.')

Parser and structured inference inspection completed without writing files.
